# NB06 — BTCS on the Libra Bank Dataset (GAP 4)
**Andre da Costa Silva | ITA | 2026**

Avaliação do BTCS no único dataset real de banco público para AML:
- **Libra Internet Bank** (Romênia), 3 meses, 385k contas, 597k arestas
- Labels: embutidos no CSV de arestas (`nr_alerts`, `nr_reports`)
- **delta_L = ∞** (sem janela temporal — grafo agregado sem timestamps)
- Referência: Dumitrescu et al., IEEE Access 2022, DOI 10.1109/ACCESS.2022.3170467

## Adaptações vs AMLSim
| Aspecto | AMLSim (nb02) | Libra Bank (nb06) |
|---|---|---|
| Labels | Edge-level (fraude por transação) | Edge-level (`nr_alerts`, `nr_reports`) |
| Timestamps | Por transação (step) | N/A — grafo agregado |
| GNN target | Edge classification | Node classification → edge scores |
| delta_L | 7 dias (AML) | ∞ (sem restrição temporal) |
| Positivos | ~1-5% das arestas | ~0.16% das arestas (alert ou report > 0) |

## Schema do CSV (arquivo único)
```
id_source, id_destination, cum_amount, nr_transactions, nr_alerts, nr_reports
```
- `nr_alerts` : nº de transações com alerta entre o par de contas
- `nr_reports` : nº de transações com relatório entre o par
- `y_edge = 1` se `nr_alerts > 0 OR nr_reports > 0`
- Node labels derivados: nó é alert/report se aparece em ≥1 aresta positiva

## Dataset path
```
GrafosGNN/data/libra/
  Libra_bank_3months_graph.csv    ← arquivo único com arestas + labels
```
(Colab: `MyDrive/GrafosGNN/data/libra/`)

Executar: células 1→2→3→4→5→6→7→8 (ou Runtime→Run all)

In [ ]:
# CELULA 1 - Setup
import os, sys, subprocess, time, math, contextlib
from pathlib import Path
from collections import defaultdict

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil', 'tqdm',
     'python-igraph', 'leidenalg')
try:
    import torch
    import torch_geometric
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    _pip('torch_geometric')
    import torch, torch_geometric

import numpy as np
import pandas as pd
import igraph as ig
import leidenalg
import psutil
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch {torch.__version__} | device: {DEVICE}')
print(f'igraph {ig.__version__} | leidenalg {leidenalg.version}')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()

# ─── Paths ────────────────────────────────────────────────
# Dataset path (arquivo único Libra_bank_3months_graph.csv)
# Priority: try several locations to handle Colab vs local
_LIBRA_CANDIDATES = [
    BASE / 'GrafosGNN/data/libra',                             # Colab + Local (padrão)
    BASE / 'DatasetDissertacao/Libra',                         # Colab (legado)
    Path(__file__).resolve().parent.parent / 'data/libra' if '__file__' in dir() else Path(''),
    Path.home() / 'Library/CloudStorage/GoogleDrive-acsilva@gmail.com/Meu Drive/GrafosGNN/data/libra',
]
_LIBRA_FILE = 'Libra_bank_3months_graph.csv'
LIBRA_DIR   = next((p for p in _LIBRA_CANDIDATES if (p / _LIBRA_FILE).exists()),
                   BASE / 'GrafosGNN/data/libra')

LIBRA_CSV   = LIBRA_DIR / _LIBRA_FILE
LIBRA_OUT   = BASE / 'GrafosGNN/results/nb06_libra'
LIBRA_OUT.mkdir(parents=True, exist_ok=True)

print(f'\nLibra dir : {LIBRA_DIR}')
print(f'CSV file  : {LIBRA_CSV}  [{"ok" if LIBRA_CSV.exists() else "MISSING"}]')
print(f'Output dir: {LIBRA_OUT}')
print('\nSetup concluido.')

In [ ]:
# CELULA 2 - Loader do Libra Bank Dataset
#
# Schema do arquivo único (Libra_bank_3months_graph.csv):
#   id_source, id_destination, cum_amount, nr_transactions, nr_alerts, nr_reports
#
# Labels de aresta  : y_edge = 1 se nr_alerts > 0 OR nr_reports > 0
# Labels de nó      : y_node = 1 se o nó aparece em ≥1 aresta positiva
# Sem arquivo separado de labels — tudo no mesmo CSV.

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb'] = proc.memory_info().rss / 1024**2 - m0


def load_libra(csv_path, verbose=True):
    """Carrega o grafo Libra a partir de um único CSV.

    Colunas esperadas:
        id_source, id_destination, cum_amount, nr_transactions, nr_alerts, nr_reports

    Retorna:
        df_edges  : DataFrame [src, dst, amount, n_tx, nr_alerts, nr_reports, y_edge]
        df_nodes  : DataFrame [node_id, alert_weight, report_weight, y_node, idx]
        node2idx  : dict str(node_id) -> int index
    """
    df_g = pd.read_csv(csv_path)
    df_g.columns = df_g.columns.str.strip().str.lower()

    if verbose:
        print(f'CSV: {df_g.shape[0]:,} arestas | cols: {list(df_g.columns)}')

    # ─── Mapeamento de colunas ───────────────────────────
    # Tenta nomes padrão Libra; fallback genérico
    src_col = next((c for c in ['id_source','src','source','from','id_src','originator']
                    if c in df_g.columns), df_g.columns[0])
    dst_col = next((c for c in ['id_destination','dst','dest','to','id_dst','beneficiary']
                    if c in df_g.columns), df_g.columns[1])
    amt_col = next((c for c in ['cum_amount','amount','weight','total_amount','value']
                    if c in df_g.columns), None)
    ntx_col = next((c for c in ['nr_transactions','n_tx','n_transactions','count','frequency']
                    if c in df_g.columns), None)
    alr_col = next((c for c in ['nr_alerts','n_alerts','alerts','alert_count','alert']
                    if c in df_g.columns), None)
    rpt_col = next((c for c in ['nr_reports','n_reports','reports','report_count','report']
                    if c in df_g.columns), None)

    if verbose:
        print(f'  src={src_col!r}  dst={dst_col!r}  amt={amt_col!r}  '
              f'n_tx={ntx_col!r}  alerts={alr_col!r}  reports={rpt_col!r}')

    # ─── Montar df_edges ────────────────────────────────
    df_edges = pd.DataFrame()
    df_edges['src_raw']   = df_g[src_col].astype(str)
    df_edges['dst_raw']   = df_g[dst_col].astype(str)
    df_edges['amount']    = (pd.to_numeric(df_g[amt_col],  errors='coerce').fillna(0)
                             if amt_col else 1.0)
    df_edges['n_tx']      = (pd.to_numeric(df_g[ntx_col],  errors='coerce').fillna(1).astype(int)
                             if ntx_col else 1)
    df_edges['nr_alerts'] = (pd.to_numeric(df_g[alr_col],  errors='coerce').fillna(0).astype(int)
                             if alr_col else 0)
    df_edges['nr_reports']= (pd.to_numeric(df_g[rpt_col],  errors='coerce').fillna(0).astype(int)
                             if rpt_col else 0)

    # Edge label: suspeita se alert OU report > 0
    df_edges['y_edge'] = (
        (df_edges['nr_alerts'] > 0) | (df_edges['nr_reports'] > 0)
    ).astype(int)

    # ─── Build node index ────────────────────────────────
    all_nodes = pd.unique(np.concatenate([
        df_edges['src_raw'].values, df_edges['dst_raw'].values]))
    node2idx = {str(n): i for i, n in enumerate(all_nodes)}
    n_nodes  = len(node2idx)

    df_edges['src'] = df_edges['src_raw'].map(node2idx).astype(int)
    df_edges['dst'] = df_edges['dst_raw'].map(node2idx).astype(int)

    # ─── Node labels (derivados das arestas) ────────────
    # Nó é "alert" se aparece como src OU dst em aresta com nr_alerts > 0
    alert_edges  = df_edges[df_edges['nr_alerts']  > 0]
    report_edges = df_edges[df_edges['nr_reports'] > 0]
    alert_nodes  = set(alert_edges['src_raw'])  | set(alert_edges['dst_raw'])
    report_nodes = set(report_edges['src_raw']) | set(report_edges['dst_raw'])

    df_nodes = pd.DataFrame({'node_id': all_nodes})
    df_nodes['alert_weight']  = df_nodes['node_id'].isin(alert_nodes).astype(int)
    df_nodes['report_weight'] = df_nodes['node_id'].isin(report_nodes).astype(int)
    df_nodes['y_node']        = ((df_nodes['alert_weight'] > 0) |
                                  (df_nodes['report_weight'] > 0)).astype(int)
    df_nodes['idx']           = df_nodes['node_id'].map(node2idx)

    if verbose:
        n_alert_nodes = int(df_nodes['y_node'].sum())
        n_pos_e       = int(df_edges['y_edge'].sum())
        print(f'\nDataset summary:')
        print(f'  Nós   : {n_nodes:,}  '
              f'(alert/report: {n_alert_nodes:,}, {100*n_alert_nodes/max(n_nodes,1):.3f}%)')
        print(f'  Arestas: {len(df_edges):,}  '
              f'(y=1: {n_pos_e:,}, {100*n_pos_e/max(len(df_edges),1):.3f}%)')
        print(f'  nr_alerts>0 : {(df_edges["nr_alerts"]>0).sum():,} arestas')
        print(f'  nr_reports>0: {(df_edges["nr_reports"]>0).sum():,} arestas')

    return df_edges, df_nodes, node2idx


print('load_libra() definido (schema: arquivo único com nr_alerts/nr_reports inline).')
print(f'Carregando {LIBRA_CSV} ...')

with measure_resources() as res_load:
    df_edges, df_nodes, node2idx = load_libra(LIBRA_CSV)

n_nodes = len(node2idx)
print(f'\nLoad concluido em {res_load["time_s"]:.1f}s '
      f'| +{res_load["ram_mb"]:.0f} MB RAM')

In [ ]:
# CELULA 3 - Node Features + GraphSAGE para Node Scoring
#
# Features por nó (edge-derived, conforme paper Dumitrescu 2022):
#   indegree, outdegree, total_in_amount, total_out_amount,
#   avg_in_amount, avg_out_amount, n_tx_in, n_tx_out
#
# GNN: GraphSAGE 2 camadas, binary node classification (y_node = alert)
# Objetivo: alto AUC na classificação de nós alert vs normal.

import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv


def build_node_features(df_edges, df_nodes, n_nodes):
    """Compute per-node structural features from edge list."""
    feats = np.zeros((n_nodes, 8), dtype=np.float32)

    for _, row in df_edges.iterrows():
        u, v = int(row['src']), int(row['dst'])
        amt  = float(row['amount'])
        ntx  = float(row['n_tx'])
        # out-features for u
        feats[u, 1] += 1          # outdegree
        feats[u, 3] += amt        # total_out_amount
        feats[u, 7] += ntx        # n_tx_out
        # in-features for v
        feats[v, 0] += 1          # indegree
        feats[v, 2] += amt        # total_in_amount
        feats[v, 6] += ntx        # n_tx_in

    # avg amounts
    safe_in  = np.maximum(feats[:, 0], 1)
    safe_out = np.maximum(feats[:, 1], 1)
    feats[:, 4] = feats[:, 2] / safe_in   # avg_in_amount
    feats[:, 5] = feats[:, 3] / safe_out  # avg_out_amount

    # Log-scale amounts to match Libra distribution (log-normal)
    feats[:, 2] = np.log1p(feats[:, 2])
    feats[:, 3] = np.log1p(feats[:, 3])
    feats[:, 4] = np.log1p(feats[:, 4])
    feats[:, 5] = np.log1p(feats[:, 5])

    # Standardize
    scaler = StandardScaler()
    feats  = scaler.fit_transform(feats).astype(np.float32)
    return feats


class GraphSAGE_Libra(torch.nn.Module):
    """2-layer GraphSAGE for node classification on Libra."""
    def __init__(self, in_channels, hidden=64, out_channels=2):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return x


def train_sage_libra(df_edges, df_nodes, n_nodes, seed=42,
                     n_epochs=100, hidden=64, lr=1e-3):
    """Treina GraphSAGE para node classification no Libra.
    Retorna node_scores: array [n_nodes] com P(alert) por nó.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    print('  Computando node features...')
    x_np = build_node_features(df_edges, df_nodes, n_nodes)

    # Node labels
    y_np = np.zeros(n_nodes, dtype=np.int64)
    for _, row in df_nodes.iterrows():
        idx = row.get('idx', None)
        if idx is not None and not pd.isna(idx):
            y_np[int(idx)] = int(row['y_node'])

    # Edge index (undirected for message passing)
    srcs = df_edges['src'].values.astype(np.int64)
    dsts = df_edges['dst'].values.astype(np.int64)
    ei   = np.stack([np.concatenate([srcs, dsts]),
                     np.concatenate([dsts, srcs])], axis=0)

    # Train/val split (80/20 stratified on labeled nodes)
    labeled_idx = df_nodes['idx'].dropna().astype(int).values
    rng = np.random.default_rng(seed)
    rng.shuffle(labeled_idx)
    n_tr = int(0.8 * len(labeled_idx))
    tr_idx = labeled_idx[:n_tr]
    va_idx = labeled_idx[n_tr:]

    train_mask = torch.zeros(n_nodes, dtype=torch.bool)
    val_mask   = torch.zeros(n_nodes, dtype=torch.bool)
    train_mask[tr_idx] = True
    val_mask[va_idx]   = True

    data = Data(
        x          = torch.tensor(x_np),
        edge_index = torch.tensor(ei, dtype=torch.long),
        y          = torch.tensor(y_np),
        train_mask = train_mask,
        val_mask   = val_mask,
    ).to(DEVICE)

    model = GraphSAGE_Libra(x_np.shape[1], hidden=hidden).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

    # Class weights for imbalance (alert nodes ~0.16%)
    n_pos = int(y_np[tr_idx].sum())
    n_neg = len(tr_idx) - n_pos
    w_pos = n_neg / max(n_pos, 1)
    weight = torch.tensor([1.0, w_pos], device=DEVICE)
    print(f'  Train: {len(tr_idx):,} nodes  pos={n_pos} neg={n_neg}  w_pos={w_pos:.1f}')

    best_auc = 0.0
    best_state = None
    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=weight)
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                probs = F.softmax(model(data.x, data.edge_index), dim=1)[:, 1].cpu().numpy()
            y_val   = y_np[va_idx]
            p_val   = probs[va_idx]
            if y_val.sum() > 0:
                auc = roc_auc_score(y_val, p_val)
                if auc > best_auc:
                    best_auc = auc
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                print(f'    Epoch {epoch+1:3d} | loss={loss.item():.4f} | val_AUC={auc:.4f}')

    # Load best
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        all_probs = F.softmax(model(data.x, data.edge_index), dim=1)[:, 1].cpu().numpy()

    print(f'  Best val AUC: {best_auc:.4f}')
    return all_probs, best_auc


print('GraphSAGE_Libra definido. Treinando...')
with measure_resources() as res_gnn:
    node_scores, gnn_auc = train_sage_libra(
        df_edges, df_nodes, n_nodes,
        seed=42, n_epochs=150, hidden=64, lr=1e-3)
print(f'GNN treinado em {res_gnn["time_s"]:.1f}s | AUC={gnn_auc:.4f}')

# ─── Edge scores: max(score_src, score_dst) ───
edge_scores = np.maximum(
    node_scores[df_edges['src'].values],
    node_scores[df_edges['dst'].values]
).astype(np.float32)

print(f'Edge scores: min={edge_scores.min():.4f} max={edge_scores.max():.4f}')
print(f'  Positive edges (y=1): {df_edges["y_edge"].sum():,}')

# AUC na tarefa de edge scoring
y_edge = df_edges['y_edge'].values
if y_edge.sum() > 0:
    edge_auc = roc_auc_score(y_edge, edge_scores)
    print(f'  Edge-level AUC: {edge_auc:.4f}')

# Salvar scores
np.save(LIBRA_OUT / 'node_scores.npy', node_scores)
np.save(LIBRA_OUT / 'edge_scores.npy', edge_scores)
print('Scores salvos.')

In [ ]:
# CELULA 4 - Adaptar estrutura para BTCS (sem timestamps)
#
# BTCS em grafo agregado: delta_L = None (sem janela temporal).
# O Lk conecta TODAS as top-k arestas que compartilham um nó —
# equivalente a delta_L = ∞.

import math
from collections import defaultdict


def build_Lk_notemporal(src_sel, dst_sel, max_hub_edges=500, verbose=True):
    """Lk sem janela temporal: conecta todas top-k arestas que compartilham nó.
    Equivalente a delta_L = ∞.
    Retorna lista de pares (min_i, max_j).
    """
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append(i)
        node_map[int(dst_sel[i])].append(i)

    lk_edges = set()
    n_hubs_capped = 0
    for node, elist in node_map.items():
        if len(elist) < 2:
            continue
        if len(elist) > max_hub_edges:
            n_hubs_capped += 1
            elist = elist[:max_hub_edges]
        for i in range(len(elist)):
            for j in range(i + 1, len(elist)):
                ei, ej = elist[i], elist[j]
                if ei != ej:
                    lk_edges.add((min(ei, ej), max(ei, ej)))
    if verbose and n_hubs_capped > 0:
        print(f'    Lk: {n_hubs_capped} hubs capped at {max_hub_edges}')
    return list(lk_edges)


def _induced_and_budget_libra(cases, sf, df_, scores, max_node, budget_B):
    """Calcula induzidas + budget cap (idêntico ao nb02/nb03)."""
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g
    g_src = np.where(sf < max_node, gid_of[sf], -1)
    g_dst = np.where(df_ < max_node, gid_of[df_], -1)
    mask  = (g_src == g_dst) & (g_src >= 0)
    idx_f = np.where(mask)[0]
    if len(idx_f) > 0:
        gf = g_src[idx_f]
        order = np.argsort(gf, kind='stable')
        g_sorted = gf[order]; i_sorted = idx_f[order]
        uq, cnt = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    n_capped = 0
    if budget_B is not None and budget_B > 0:
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx_arr = np.array(case['induced_edges'], dtype=np.int64)
                sc_ind  = scores[idx_arr]
                top_b   = idx_arr[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()
    return cases, n_capped


def build_btcs_libra(df_edges_data, k=0.01, resolution=1.0, budget_B=100, seed=42):
    """BTCS no grafo Libra (delta_L = ∞, sem timestamps).

    df_edges_data: dict com keys 'scores', 'src', 'dst', 'y'
    """
    scores = np.asarray(df_edges_data['scores'], dtype=float)
    src    = np.asarray(df_edges_data['src'],    dtype=np.int64)
    dst    = np.asarray(df_edges_data['dst'],    dtype=np.int64)
    sf     = src  # full graph = test graph (Libra é não dividido)
    df_    = dst
    K      = max(1, int(math.ceil(k * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_sel, dst_sel = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1
    print(f'  K={K:,} | mode=wcc_leiden | delta_L=∞ | B={budget_B}')

    # ─── WCC no subgrafo top-k ───
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    n_compact = len(all_nodes)
    edges_compact = [(nmap[int(s)], nmap[int(d)]) for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=n_compact, edges=edges_compact, directed=False)
    g_node.simplify()
    wcc     = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc   = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)

    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets  = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])

    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes: wcc_gid[nid] = w
    g_src_w = np.where(sf < max_node, wcc_gid[sf], -1)
    g_dst_w = np.where(df_ < max_node, wcc_gid[df_], -1)
    mask_w  = (g_src_w == g_dst_w) & (g_src_w >= 0)
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if mask_w.sum() > 0:
        gw = g_src_w[np.where(mask_w)[0]]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt

    # ─── WCC+Leiden hierárquico ───
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0; n_kept = 0; n_split = 0

    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp: continue
        need_split = (budget_B is not None
                      and wcc_ind_count[w] > budget_B and len(comp) >= 2)
        if not need_split:
            for i in comp: final_mem[i] = next_id
            next_id += 1; n_kept += 1
        else:
            n_split += 1
            comp_arr = np.array(comp, dtype=np.int64)
            # Build Lk sem temporal (delta_L = ∞)
            lk_local = build_Lk_notemporal(
                src_sel[comp_arr], dst_sel[comp_arr], verbose=False)
            if not lk_local:
                for i in comp: final_mem[i] = next_id; next_id += 1
            else:
                n_local = len(comp)
                g_local = ig.Graph(n=n_local, edges=lk_local, directed=False)
                g_local.simplify()
                part = leidenalg.find_partition(
                    g_local, leidenalg.RBConfigurationVertexPartition,
                    resolution_parameter=resolution, seed=seed)
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub

    n_total = next_id
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0: continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0: cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    cases, n_capped = _induced_and_budget_libra(cases, sf, df_, scores, max_node, budget_B)
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  WCC+Leiden: {n_kept} kept + {n_split} split → {len(cases)} cases | '
          f'capped={n_capped} | median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases


print('build_btcs_libra() (delta_L=∞) definido.')

In [ ]:
# CELULA 5 - Métricas + Baselines para Libra

def evaluate_cases_libra(cases, y_edge_full, n_pos_total):
    """Métricas BTCS adaptadas para Libra (sem split train/test)."""
    if not cases:
        return {m: np.nan for m in ['n_cases','coverage','purity_induced',
                                     'ocr_b100','edges_per_case_median',
                                     'edges_per_case_max','e_ind_total']}
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    pos_ind  = int(y_edge_full[all_ind].sum()) if len(all_ind) > 0 else 0
    coverage = float(pos_ind / max(n_pos_total, 1))
    purity   = float(pos_ind / max(int(ind_sizes.sum()), 1))
    return {'n_cases': len(cases), 'coverage': coverage,
            'purity_induced': purity,
            'ocr_b100': float((ind_sizes > 100).mean()),
            'edges_per_case_median': float(np.median(ind_sizes)),
            'edges_per_case_max':    float(ind_sizes.max()),
            'e_ind_total':           int(ind_sizes.sum())}


def run_wcc_libra(data, budget_B=100):
    """B1: WCC-only com budget cap."""
    scores = np.asarray(data['scores'], dtype=float)
    src    = np.asarray(data['src'], dtype=np.int64)
    dst    = np.asarray(data['dst'], dtype=np.int64)
    K      = max(1, int(math.ceil(0.01 * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_s, dst_s = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    all_n = np.unique(np.concatenate([src_s, dst_s]))
    nmap  = {int(n): i for i, n in enumerate(all_n)}
    g     = ig.Graph(n=len(all_n),
                     edges=[(nmap[int(s)], nmap[int(d)]) for s, d in zip(src_s, dst_s)],
                     directed=False)
    g.simplify()
    wcc = g.connected_components(mode='weak')
    mem = np.array(wcc.membership, dtype=np.int64)
    n_w = int(mem.max()) + 1
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_w)]
    for i in range(K):
        w = int(mem[nmap[int(src_s[i])]])
        cases[w]['nodes'].update([int(src_s[i]), int(dst_s[i])])
        cases[w]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    cases, _ = _induced_and_budget_libra(cases, src, dst, scores, max_node, budget_B)
    return cases


def run_louvain_libra(data, budget_B=100, seed=42):
    """B2: Louvain + truncamento."""
    scores = np.asarray(data['scores'], dtype=float)
    src    = np.asarray(data['src'], dtype=np.int64)
    dst    = np.asarray(data['dst'], dtype=np.int64)
    K      = max(1, int(math.ceil(0.01 * len(scores))))
    sel    = np.argsort(-scores)[:K]
    src_s, dst_s = src[sel], dst[sel]
    max_node = int(max(src.max(), dst.max())) + 1

    lk_edges = build_Lk_notemporal(src_s, dst_s, verbose=False)
    g_lk     = ig.Graph(n=K, edges=lk_edges, directed=False)
    g_lk.simplify()
    clusters = g_lk.community_multilevel()
    mem      = np.array(clusters.membership, dtype=np.int64)
    n_c      = int(mem.max()) + 1
    cases    = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_c)]
    for i in range(K):
        g = int(mem[i])
        cases[g]['nodes'].update([int(src_s[i]), int(dst_s[i])])
        cases[g]['seed_edges'].append(int(sel[i]))
    cases    = [c for c in cases if c['nodes']]
    cases, _ = _induced_and_budget_libra(cases, src, dst, scores, max_node, budget_B)
    return cases


print('Funções de baseline e avaliação definidas.')

In [ ]:
# CELULA 6 - Execução: BTCS + Baselines no Libra

y_edge_arr = df_edges['y_edge'].values.astype(int)
n_pos_total = int(y_edge_arr.sum())
n_edges     = len(df_edges)

data_libra = {
    'scores': edge_scores,
    'src':    df_edges['src'].values.astype(np.int64),
    'dst':    df_edges['dst'].values.astype(np.int64),
    'y':      y_edge_arr,
}

K_1pct = max(1, int(math.ceil(0.01 * n_edges)))
print(f'Libra: {n_edges:,} arestas | {n_pos_total:,} positivas ({100*n_pos_total/n_edges:.3f}%)')
print(f'k=1%: K={K_1pct:,} arestas selecionadas')

rows = []
K_VALS = [0.005, 0.01, 0.02]  # k = 0.5%, 1%, 2% (pos. rate ~0.16% então 1%>> positivos)

METHODS = [
    ('BTCS_v3',   None),   # BTCS WCC+Leiden
    ('B1_WCC',    'wcc'),
    ('B2_Louvain','louvain'),
]

for k in K_VALS:
    kp = f'{k*100:.1f}%'
    print(f'\n── k={kp} ──')

    # BTCS
    with measure_resources() as res:
        cases_btcs = build_btcs_libra(data_libra, k=k, budget_B=100)
    m = evaluate_cases_libra(cases_btcs, y_edge_arr, n_pos_total)
    rows.append({'method':'BTCS_v3', 'k%':kp, **m, **res})
    print(f'  BTCS: cov={m["coverage"]:.3f} pur={m["purity_induced"]:.4f} '
          f'OCR={m["ocr_b100"]:.3f} #cases={m["n_cases"]:,} {res["time_s"]:.1f}s')

    # WCC baseline
    with measure_resources() as res:
        data_k = dict(data_libra)
        data_k['k'] = k
        scores_arr = np.asarray(data_libra['scores'], dtype=float)
        src_arr    = np.asarray(data_libra['src'], dtype=np.int64)
        dst_arr    = np.asarray(data_libra['dst'], dtype=np.int64)
        K_actual   = max(1, int(math.ceil(k * len(scores_arr))))
        sel_k      = np.argsort(-scores_arr)[:K_actual]
        max_node   = int(max(src_arr.max(), dst_arr.max())) + 1
        src_k, dst_k = src_arr[sel_k], dst_arr[sel_k]
        all_n = np.unique(np.concatenate([src_k, dst_k]))
        nmap_k = {int(n): i for i, n in enumerate(all_n)}
        g_wcc = ig.Graph(n=len(all_n),
                         edges=[(nmap_k[int(s)], nmap_k[int(d)])
                                for s, d in zip(src_k, dst_k)],
                         directed=False)
        g_wcc.simplify()
        wcc_c = g_wcc.connected_components(mode='weak')
        mem_k = np.array(wcc_c.membership, dtype=np.int64)
        n_wk  = int(mem_k.max()) + 1
        cases_wcc = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(n_wk)]
        for i in range(K_actual):
            w = int(mem_k[nmap_k[int(src_k[i])]])
            cases_wcc[w]['nodes'].update([int(src_k[i]), int(dst_k[i])])
            cases_wcc[w]['seed_edges'].append(int(sel_k[i]))
        cases_wcc = [c for c in cases_wcc if c['nodes']]
        cases_wcc, _ = _induced_and_budget_libra(cases_wcc, src_arr, dst_arr, scores_arr, max_node, 100)
    m_wcc = evaluate_cases_libra(cases_wcc, y_edge_arr, n_pos_total)
    rows.append({'method':'B1_WCC', 'k%':kp, **m_wcc, **res})
    print(f'  WCC:  cov={m_wcc["coverage"]:.3f} pur={m_wcc["purity_induced"]:.4f} '
          f'OCR={m_wcc["ocr_b100"]:.3f} #cases={m_wcc["n_cases"]:,} {res["time_s"]:.1f}s')

    # Louvain baseline
    with measure_resources() as res:
        lk_e = build_Lk_notemporal(src_k, dst_k, verbose=False)
        g_lk = ig.Graph(n=K_actual, edges=lk_e, directed=False)
        g_lk.simplify()
        cl   = g_lk.community_multilevel()
        mL   = np.array(cl.membership, dtype=np.int64)
        nL   = int(mL.max()) + 1
        cases_lou = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []} for _ in range(nL)]
        for i in range(K_actual):
            g = int(mL[i])
            cases_lou[g]['nodes'].update([int(src_k[i]), int(dst_k[i])])
            cases_lou[g]['seed_edges'].append(int(sel_k[i]))
        cases_lou = [c for c in cases_lou if c['nodes']]
        cases_lou, _ = _induced_and_budget_libra(cases_lou, src_arr, dst_arr, scores_arr, max_node, 100)
    m_lou = evaluate_cases_libra(cases_lou, y_edge_arr, n_pos_total)
    rows.append({'method':'B2_Louvain', 'k%':kp, **m_lou, **res})
    print(f'  Louv: cov={m_lou["coverage"]:.3f} pur={m_lou["purity_induced"]:.4f} '
          f'OCR={m_lou["ocr_b100"]:.3f} #cases={m_lou["n_cases"]:,} {res["time_s"]:.1f}s')

df_libra_res = pd.DataFrame(rows)
print('\n=== Resultados Libra Bank ===')
pivot = df_libra_res.pivot_table(
    index=['k%','method'],
    values=['coverage','purity_induced','ocr_b100','n_cases','time_s'],
    aggfunc='first').round(4)
display(pivot)

In [ ]:
# CELULA 7 - Plot: BTCS vs Baselines no Libra
import matplotlib.pyplot as plt
import numpy as np

metrics = ['coverage', 'purity_induced', 'ocr_b100']
mlabels = ['Coverage', 'Purity (Induced)', 'OCR(B=100)']
methods = ['BTCS_v3', 'B1_WCC', 'B2_Louvain']
m_colors = {'BTCS_v3':'#2ca02c', 'B1_WCC':'#1f77b4', 'B2_Louvain':'#ff7f0e'}
m_labels = {'BTCS_v3':'BTCS v3 (WCC+Leiden)', 'B1_WCC':'WCC-only', 'B2_Louvain':'Louvain'}
k_vals   = sorted(df_libra_res['k%'].unique())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x = np.arange(len(k_vals)); w = 0.25

for col, (metric, mlabel) in enumerate(zip(metrics, mlabels)):
    ax = axes[col]
    for j, meth in enumerate(methods):
        vals = []
        for kp in k_vals:
            row = df_libra_res[(df_libra_res['method']==meth) & (df_libra_res['k%']==kp)]
            vals.append(float(row[metric]) if not row.empty else 0.0)
        bars = ax.bar(x + j*w, vals, w, label=m_labels[meth],
                      color=m_colors[meth], alpha=0.85)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
                    f'{v:.3f}', ha='center', va='bottom', fontsize=7)
    ax.set_title(f'{mlabel} — Libra Bank (Real)', fontsize=11)
    ax.set_xticks(x + w); ax.set_xticklabels([f'k={kp}' for kp in k_vals], fontsize=9)
    ax.set_ylabel(mlabel); ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

plt.suptitle('BTCS vs Baselines — Libra Internet Bank (Romênia)\n'
             'delta_L=∞ (aggregated graph), B=100, GNN=GraphSAGE node scores',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(LIBRA_OUT / 'libra_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot salvo: {LIBRA_OUT}/libra_results.png')

In [ ]:
# CELULA 8 - Export + Resumo
df_libra_res.to_csv(LIBRA_OUT / 'libra_btcs_results.csv', index=False)

export_cols = ['method','k%','n_cases','coverage','purity_induced','ocr_b100','time_s']
df_libra_res[export_cols].round(4).to_latex(
    LIBRA_OUT / 'libra_btcs_results.tex', index=False, escape=False)

print(f'CSV: {LIBRA_OUT}/libra_btcs_results.csv')
print(f'TEX: {LIBRA_OUT}/libra_btcs_results.tex')

# ─── Resumo para paper ───
print('\n=== RESUMO PARA O PAPER ===')
print(f'Dataset: Libra Internet Bank (Romênia, real)')
print(f'  {n_edges:,} arestas  |  {n_nodes:,} nós  |  {n_pos_total:,} arestas positivas (alert)')
print(f'  GNN AUC (node): {gnn_auc:.4f}')
print(f'  delta_L = ∞ (grafo agregado, sem timestamps)')
print(f'  B = 100  |  k = 1%')
print()
for _, row in df_libra_res[df_libra_res['k%']=='1.0%'].iterrows():
    print(f'  {row["method"]:15s}: cov={row["coverage"]:.3f}  '
          f'pur={row["purity_induced"]:.4f}  OCR={row["ocr_b100"]:.3f}  '
          f'#cases={row["n_cases"]:,}')

print('\nProgresso:')
print('1.[ok] nb00  2.[ok] nb01  3.[v3] nb02  4.[ok] nb03  5.[ok] nb04')
print('6.[ok] nb05-GNN  7.[ok] nb06-Libra (GAP 4 FECHADO)')